# Options Strategy Recommendation and Optimisation System
## Notebook 1 — Data Collection and Preparation

This notebook builds a reusable, direction-neutral market dataset for either one ticker or the current S&P 500 constituent list.

### Revised batch design
1. Choose `RUN_MODE = "single"` while testing or `RUN_MODE = "sp500"` for the full universe.
2. Download price history and current option chains once per ticker.
3. Retain both calls and puts so the same dataset can support bullish and bearish reasoning later.
4. Save every ticker under its own folder to prevent overwriting.
5. Record failures and continue with the remaining tickers.
6. Resume a partial batch by skipping tickers with a completion marker.

> Outlook is deliberately not an input to Notebook 1. Bullish/bearish strategy selection belongs in Notebook 3 and later notebooks.


## Important data limitation

`yfinance` provides the current listed option chain, not a complete point-in-time archive of expired chains. These outputs support current recommendations. A fair historical backtest must use reconstructed/theoretical option prices or a proper historical options dataset and must not assume that today's chain existed in the past.


## 1. Environment setup

Install packages once from a terminal with:

`python -m pip install yfinance pandas numpy matplotlib lxml html5lib requests certifi`


In [ ]:
from __future__ import annotations

import json
from io import StringIO
import math
import re
import sys
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from statistics import NormalDist
from typing import Any

import certifi
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import yfinance as yf
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

print(f"Python: {sys.version.split()[0]} | pandas: {pd.__version__} | yfinance: {yf.__version__}")


## 2. Batch parameters

Use `single` mode first. When it succeeds, change `RUN_MODE` to `sp500`. The cell is tagged `parameters` so a future Papermill runner can override these values.


In [ ]:
# Execution mode
RUN_MODE = "single"              # allowed: "single", "sp500"
TICKERS = ["CRM"]                # used only in single mode
MAX_TICKERS = None               # e.g. 5 for a small S&P 500 test

# Batch controls
OUTPUT_ROOT = "outputs/notebook_01"
SKIP_COMPLETED_TICKERS = True
REQUEST_PAUSE_SECONDS = 1.0
DISPLAY_SAMPLE = True

# Market-data defaults
HISTORY_PERIOD = "2y"
MIN_DTE = 21
MAX_DTE = 60
MIN_OPEN_INTEREST = 50
MAX_BID_ASK_SPREAD_PCT = 0.30
MIN_STRIKE_TO_SPOT = 0.50
MAX_STRIKE_TO_SPOT = 1.50
FALLBACK_RISK_FREE_RATE = 0.04
RISK_FREE_TICKER = "^IRX"

# Current constituent source
SP500_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"


In [ ]:
OUTPUT_ROOT = Path(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

RUN_MODE = RUN_MODE.strip().lower()
if RUN_MODE not in {"single", "sp500"}:
    raise ValueError("RUN_MODE must be either 'single' or 'sp500'.")
if MIN_DTE < 1 or MAX_DTE < MIN_DTE:
    raise ValueError("DTE range must satisfy 1 <= MIN_DTE <= MAX_DTE.")
if MAX_TICKERS is not None and MAX_TICKERS < 1:
    raise ValueError("MAX_TICKERS must be None or a positive integer.")

@dataclass(frozen=True)
class MarketDataRequest:
    ticker: str
    min_dte: int = MIN_DTE
    max_dte: int = MAX_DTE

def validate_ticker(ticker: str) -> str:
    clean_ticker = str(ticker).strip().upper().replace(".", "-")
    if not clean_ticker or not re.fullmatch(r"[A-Z0-9^=-]{1,15}", clean_ticker):
        raise ValueError(f"Unsupported ticker: {ticker!r}")
    return clean_ticker

def fetch_sp500_tickers(url: str = SP500_URL) -> pd.DataFrame:
    """Download the current S&P 500 table using certifi's CA bundle."""
    try:
        response = requests.get(
            url,
            timeout=30,
            headers={"User-Agent": "Mozilla/5.0 options-research-project"},
            verify=certifi.where(),
        )
        response.raise_for_status()
    except requests.RequestException as error:
        raise RuntimeError(
            "Unable to download the S&P 500 constituent page using the certifi "
            "certificate bundle. Confirm Internet access and reinstall requests/certifi."
        ) from error

    tables = pd.read_html(StringIO(response.text))
    if not tables or "Symbol" not in tables[0].columns:
        raise ValueError("The S&P 500 constituent table could not be identified.")

    constituents = tables[0].copy()
    constituents["Yahoo_Symbol"] = (
        constituents["Symbol"].astype(str).str.strip().str.upper()
        .str.replace(".", "-", regex=False)
    )
    constituents = constituents.drop_duplicates("Yahoo_Symbol").reset_index(drop=True)
    return constituents

def resolve_ticker_universe() -> tuple[list[str], pd.DataFrame | None]:
    if RUN_MODE == "single":
        tickers = list(dict.fromkeys(validate_ticker(ticker) for ticker in TICKERS))
        constituents = None
    else:
        constituents = fetch_sp500_tickers()
        tickers = constituents["Yahoo_Symbol"].map(validate_ticker).tolist()

    if MAX_TICKERS is not None:
        tickers = tickers[:MAX_TICKERS]
        if constituents is not None:
            constituents = constituents.loc[
                constituents["Yahoo_Symbol"].isin(tickers)
            ].copy()

    if not tickers:
        raise ValueError("No tickers were selected.")
    return tickers, constituents


## 3. Reusable market-data functions


In [ ]:
def flatten_single_ticker_columns(frame: pd.DataFrame, ticker: str) -> pd.DataFrame:
    """Return conventional OHLCV columns for one yfinance ticker."""
    result = frame.copy()
    if isinstance(result.columns, pd.MultiIndex):
        if ticker in result.columns.get_level_values(-1):
            result = result.xs(ticker, axis=1, level=-1)
        else:
            result.columns = result.columns.get_level_values(0)
    result.columns.name = None
    return result

def fetch_stock_history(ticker: str, period: str = HISTORY_PERIOD) -> pd.DataFrame:
    history = yf.download(
        ticker,
        period=period,
        interval="1d",
        auto_adjust=False,
        progress=False,
        threads=False,
    )
    history = flatten_single_ticker_columns(history, ticker)
    if history.empty or "Close" not in history.columns:
        raise ValueError(f"No historical price data returned for {ticker}.")
    history = history.sort_index()
    history.index = pd.to_datetime(history.index)
    return history

def add_market_features(history: pd.DataFrame) -> pd.DataFrame:
    result = history.copy()
    close = pd.to_numeric(result["Close"], errors="coerce")
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    rs = gain / loss.replace(0, np.nan)

    result["daily_return"] = close.pct_change()
    result["log_return"] = np.log(close / close.shift(1))
    result["sma_20"] = close.rolling(20).mean()
    result["sma_50"] = close.rolling(50).mean()
    result["realized_vol_20"] = result["log_return"].rolling(20).std() * np.sqrt(252)
    result["realized_vol_60"] = result["log_return"].rolling(60).std() * np.sqrt(252)
    result["rsi_14"] = 100 - (100 / (1 + rs))
    result["momentum_20"] = close.pct_change(20)
    result["trend_state"] = np.select(
        [close > result["sma_50"], close < result["sma_50"]],
        ["above_sma50", "below_sma50"],
        default="neutral",
    )
    return result

def fetch_risk_free_rate(
    fallback: float = FALLBACK_RISK_FREE_RATE,
) -> tuple[float, str]:
    try:
        rates = yf.download(
            RISK_FREE_TICKER,
            period="1mo",
            interval="1d",
            auto_adjust=False,
            progress=False,
            threads=False,
        )
        rates = flatten_single_ticker_columns(rates, RISK_FREE_TICKER)
        value = float(pd.to_numeric(rates["Close"], errors="coerce").dropna().iloc[-1]) / 100
        if not 0 <= value <= 0.25:
            raise ValueError("Risk-free proxy is outside a reasonable range.")
        return value, RISK_FREE_TICKER
    except Exception as error:
        print(f"Risk-free rate fallback used ({fallback:.2%}): {error}")
        return fallback, "fallback"

def estimate_dividend_yield(stock: yf.Ticker, spot: float) -> tuple[float, str]:
    try:
        dividends = stock.dividends
        if dividends.empty:
            return 0.0, "no_recent_dividends"
        cutoff = pd.Timestamp.now(tz=dividends.index.tz) - pd.Timedelta(days=365)
        trailing_dividend = float(dividends.loc[dividends.index >= cutoff].sum())
        value = trailing_dividend / spot
        return (value, "trailing_12m") if 0 <= value <= 0.20 else (0.0, "invalid_fallback")
    except Exception as error:
        print(f"Dividend yield fallback used (0.00%): {error}")
        return 0.0, "fallback"

def select_expiries(stock: yf.Ticker, min_dte: int, max_dte: int) -> pd.DataFrame:
    today = pd.Timestamp.now(tz="UTC").normalize().tz_localize(None)
    rows = []
    for expiry_text in stock.options:
        expiry = pd.Timestamp(expiry_text)
        dte = int((expiry - today).days)
        if dte > 0:
            rows.append({"expiry": expiry_text, "dte": dte})

    expiries = pd.DataFrame(rows, columns=["expiry", "dte"])
    if expiries.empty:
        raise ValueError("No future listed option expiries were returned.")
    expiries = expiries.sort_values("dte").reset_index(drop=True)

    selected = expiries.loc[expiries["dte"].between(min_dte, max_dte)].copy()
    if selected.empty:
        target_dte = (min_dte + max_dte) / 2
        nearest_index = (expiries["dte"] - target_dte).abs().idxmin()
        selected = expiries.loc[[nearest_index]].copy()
        print("No expiry fell inside the DTE range; the closest future expiry was selected.")
    return selected.reset_index(drop=True)

def fetch_option_chains(
    stock: yf.Ticker,
    selected_expiries: pd.DataFrame,
) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    errors: list[str] = []

    for row in selected_expiries.itertuples(index=False):
        try:
            chain = stock.option_chain(row.expiry)
            for option_type, contracts in (("call", chain.calls), ("put", chain.puts)):
                item = contracts.copy()
                item["option_type"] = option_type
                item["expiry"] = row.expiry
                item["dte"] = int(row.dte)
                frames.append(item)
        except Exception as error:
            errors.append(f"{row.expiry}: {error}")

    if not frames:
        raise ValueError("No option chains could be downloaded. " + " | ".join(errors))
    if errors:
        print("Some expiries could not be downloaded:", *errors, sep="\n- ")
    return pd.concat(frames, ignore_index=True)


In [ ]:
def normal_pdf(value: float) -> float:
    return math.exp(-0.5 * value * value) / math.sqrt(2 * math.pi)

def black_scholes_greeks(
    spot: float,
    strike: float,
    ttm: float,
    rate: float,
    dividend_yield: float,
    volatility: float,
    option_type: str,
) -> dict[str, float]:
    if min(spot, strike, ttm, volatility) <= 0:
        return {name: np.nan for name in ("delta", "gamma", "theta_day", "vega_1pct")}

    sqrt_t = math.sqrt(ttm)
    d1 = (
        math.log(spot / strike)
        + (rate - dividend_yield + 0.5 * volatility**2) * ttm
    ) / (volatility * sqrt_t)
    d2 = d1 - volatility * sqrt_t
    cdf = NormalDist().cdf
    discounted_dividend = math.exp(-dividend_yield * ttm)
    discounted_strike = math.exp(-rate * ttm)

    gamma = discounted_dividend * normal_pdf(d1) / (spot * volatility * sqrt_t)
    vega = spot * discounted_dividend * normal_pdf(d1) * sqrt_t / 100
    common_theta = -(
        spot * discounted_dividend * normal_pdf(d1) * volatility
    ) / (2 * sqrt_t)

    if option_type == "call":
        delta = discounted_dividend * cdf(d1)
        theta = (
            common_theta
            - rate * strike * discounted_strike * cdf(d2)
            + dividend_yield * spot * discounted_dividend * cdf(d1)
        )
    else:
        delta = discounted_dividend * (cdf(d1) - 1)
        theta = (
            common_theta
            + rate * strike * discounted_strike * cdf(-d2)
            - dividend_yield * spot * discounted_dividend * cdf(-d1)
        )

    return {
        "delta": delta,
        "gamma": gamma,
        "theta_day": theta / 365,
        "vega_1pct": vega,
    }

def prepare_option_chain(
    raw: pd.DataFrame,
    spot: float,
    rate: float,
    dividend_yield: float,
) -> pd.DataFrame:
    result = raw.copy()
    numeric_columns = [
        "strike", "lastPrice", "bid", "ask", "change", "percentChange",
        "volume", "openInterest", "impliedVolatility", "dte",
    ]
    for column in numeric_columns:
        if column in result.columns:
            result[column] = pd.to_numeric(result[column], errors="coerce")

    valid_quote = (
        (result["bid"] >= 0)
        & (result["ask"] > 0)
        & (result["ask"] >= result["bid"])
    )
    result["mid_price"] = np.where(
        valid_quote,
        (result["bid"] + result["ask"]) / 2,
        result["lastPrice"],
    )
    result["bid_ask_spread"] = np.where(
        valid_quote, result["ask"] - result["bid"], np.nan
    )
    result["bid_ask_spread_pct"] = (
        result["bid_ask_spread"] / result["mid_price"].replace(0, np.nan)
    )
    result["spot_price"] = spot
    result["strike_to_spot"] = result["strike"] / spot
    result["moneyness_pct"] = (result["strike"] - spot) / spot
    result["absolute_moneyness_pct"] = result["moneyness_pct"].abs()
    result["intrinsic_value"] = np.where(
        result["option_type"].eq("call"),
        np.maximum(spot - result["strike"], 0),
        np.maximum(result["strike"] - spot, 0),
    )
    result["time_value"] = (
        result["mid_price"] - result["intrinsic_value"]
    ).clip(lower=0)
    result["ttm_years"] = result["dte"].clip(lower=1) / 365

    greeks = result.apply(
        lambda row: black_scholes_greeks(
            spot=spot,
            strike=row["strike"],
            ttm=row["ttm_years"],
            rate=rate,
            dividend_yield=dividend_yield,
            volatility=row["impliedVolatility"],
            option_type=row["option_type"],
        ),
        axis=1,
        result_type="expand",
    )
    result = pd.concat([result, greeks], axis=1)

    result["liquidity_pass"] = (
        result["openInterest"].fillna(0).ge(MIN_OPEN_INTEREST)
        & result["bid_ask_spread_pct"].le(MAX_BID_ASK_SPREAD_PCT)
        & result["mid_price"].gt(0)
    )
    result["valid_iv"] = result["impliedVolatility"].between(0.01, 5.0)
    return result.sort_values(
        ["expiry", "option_type", "strike"]
    ).reset_index(drop=True)

def finite_or_none(value: Any) -> float | None:
    try:
        numeric = float(value)
        return numeric if math.isfinite(numeric) else None
    except (TypeError, ValueError):
        return None


## 4. Per-ticker collection pipeline

Every ticker writes to `outputs/notebook_01/<TICKER>/`. There is no outlook in these paths because the underlying data is shared by bullish and bearish runs.


In [ ]:
def expected_export_paths(ticker: str) -> dict[str, Path]:
    ticker_dir = OUTPUT_ROOT / ticker
    return {
        "history_features": ticker_dir / "history_features.csv",
        "option_chain_raw": ticker_dir / "option_chain_raw.csv",
        "option_universe": ticker_dir / "option_universe.csv",
        "market_snapshot": ticker_dir / "market_snapshot.json",
        "quality_report": ticker_dir / "quality_report.csv",
        "success_marker": ticker_dir / "_SUCCESS.json",
    }

def collect_ticker_dataset(
    ticker: str,
    risk_free_rate: float,
    risk_free_source: str,
) -> dict[str, Any]:
    ticker = validate_ticker(ticker)
    request = MarketDataRequest(ticker=ticker)
    export_paths = expected_export_paths(ticker)
    ticker_dir = export_paths["success_marker"].parent
    ticker_dir.mkdir(parents=True, exist_ok=True)

    stock = yf.Ticker(ticker)
    history_raw = fetch_stock_history(ticker)
    history_features = add_market_features(history_raw)
    valid_close = pd.to_numeric(history_features["Close"], errors="coerce").dropna()
    if valid_close.empty:
        raise ValueError("No valid closing price was available.")

    spot_price = float(valid_close.iloc[-1])
    dividend_yield, dividend_yield_source = estimate_dividend_yield(
        stock, spot_price
    )
    selected_expiries = select_expiries(stock, request.min_dte, request.max_dte)
    option_chain_raw = fetch_option_chains(stock, selected_expiries)
    option_chain_prepared = prepare_option_chain(
        option_chain_raw,
        spot_price,
        risk_free_rate,
        dividend_yield,
    )

    base_filter = (
        option_chain_prepared["strike_to_spot"].between(
            MIN_STRIKE_TO_SPOT, MAX_STRIKE_TO_SPOT
        )
        & option_chain_prepared["valid_iv"]
        & option_chain_prepared["mid_price"].gt(0)
    )
    strict_filter = base_filter & option_chain_prepared["liquidity_pass"]
    option_universe = option_chain_prepared.loc[strict_filter].copy()
    filter_mode = "strict_liquidity"

    if option_universe.empty:
        option_universe = option_chain_prepared.loc[base_filter].copy()
        filter_mode = "relaxed_liquidity_fallback"
    if option_universe.empty:
        raise ValueError("No usable option contracts remain after validation.")

    latest = history_features.dropna(subset=["Close"]).iloc[-1]
    market_signal = (
        "bullish"
        if latest.get("Close", np.nan) > latest.get("sma_50", np.nan)
        else "bearish"
        if latest.get("Close", np.nan) < latest.get("sma_50", np.nan)
        else "neutral"
    )

    market_snapshot: dict[str, Any] = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "request": asdict(request),
        "market": {
            "spot_price": spot_price,
            "risk_free_rate": risk_free_rate,
            "risk_free_source": risk_free_source,
            "dividend_yield": dividend_yield,
            "dividend_yield_source": dividend_yield_source,
            "realized_vol_20": finite_or_none(latest.get("realized_vol_20")),
            "realized_vol_60": finite_or_none(latest.get("realized_vol_60")),
            "rsi_14": finite_or_none(latest.get("rsi_14")),
            "momentum_20": finite_or_none(latest.get("momentum_20")),
            "trend_state": str(latest.get("trend_state", "unknown")),
            "technical_signal": market_signal,
        },
        "option_data": {
            "selected_expiries": selected_expiries.to_dict(orient="records"),
            "raw_contract_count": int(len(option_chain_raw)),
            "usable_contract_count": int(len(option_universe)),
            "filter_mode": filter_mode,
            "contains_calls_and_puts": True,
        },
        "assumptions": {
            "contract_multiplier": 100,
            "pricing_basis": "mid price when bid/ask is valid; otherwise last price",
            "greek_model": "Black-Scholes with trailing dividend yield",
            "directional_outlook_applied": False,
        },
    }

    quality_checks = {
        "history_has_at_least_60_rows": len(history_features) >= 60,
        "spot_price_is_positive": np.isfinite(spot_price) and spot_price > 0,
        "future_expiry_available": bool((selected_expiries["dte"] > 0).all()),
        "option_universe_is_not_empty": not option_universe.empty,
        "contract_symbols_are_unique": not option_universe["contractSymbol"].duplicated().any(),
        "all_mid_prices_are_positive": bool(option_universe["mid_price"].gt(0).all()),
        "all_dte_values_are_positive": bool(option_universe["dte"].gt(0).all()),
        "all_option_types_are_valid": bool(option_universe["option_type"].isin(["call", "put"]).all()),
        "calls_are_available": bool(option_universe["option_type"].eq("call").any()),
        "puts_are_available": bool(option_universe["option_type"].eq("put").any()),
    }
    quality_report = pd.DataFrame(
        [{"check": name, "passed": passed} for name, passed in quality_checks.items()]
    )
    failed_checks = quality_report.loc[
        ~quality_report["passed"], "check"
    ].tolist()
    if failed_checks:
        raise AssertionError(f"Data-quality checks failed: {failed_checks}")

    history_features.to_csv(export_paths["history_features"], index_label="Date")
    option_chain_raw.to_csv(export_paths["option_chain_raw"], index=False)
    option_universe.to_csv(export_paths["option_universe"], index=False)
    quality_report.to_csv(export_paths["quality_report"], index=False)
    with export_paths["market_snapshot"].open("w", encoding="utf-8") as file:
        json.dump(market_snapshot, file, indent=2, allow_nan=False)

    completion = {
        "ticker": ticker,
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        "usable_contract_count": int(len(option_universe)),
    }
    with export_paths["success_marker"].open("w", encoding="utf-8") as file:
        json.dump(completion, file, indent=2)

    return {
        "ticker": ticker,
        "status": "completed",
        "spot_price": spot_price,
        "raw_contracts": int(len(option_chain_raw)),
        "usable_contracts": int(len(option_universe)),
        "filter_mode": filter_mode,
        "technical_signal": market_signal,
        "output_directory": str(ticker_dir),
        "error": None,
    }


## 5. Run the selected universe

The risk-free proxy is fetched once for the complete batch. A failed ticker is recorded rather than stopping all remaining runs.


In [ ]:
tickers, constituents = resolve_ticker_universe()

if constituents is not None:
    constituents_path = OUTPUT_ROOT / "sp500_constituents.csv"
    constituents.to_csv(constituents_path, index=False)
    print(f"Saved constituent snapshot: {constituents_path}")

print(f"Run mode: {RUN_MODE}")
print(f"Tickers selected: {len(tickers):,}")

risk_free_rate, risk_free_source = fetch_risk_free_rate()
print(f"Risk-free rate: {risk_free_rate:.2%} ({risk_free_source})")

batch_rows: list[dict[str, Any]] = []

for position, ticker in enumerate(tickers, start=1):
    export_paths = expected_export_paths(ticker)
    print(f"[{position}/{len(tickers)}] {ticker}")

    if SKIP_COMPLETED_TICKERS and export_paths["success_marker"].exists():
        batch_rows.append({
            "ticker": ticker,
            "status": "skipped_existing",
            "spot_price": None,
            "raw_contracts": None,
            "usable_contracts": None,
            "filter_mode": None,
            "technical_signal": None,
            "output_directory": str(export_paths["success_marker"].parent),
            "error": None,
        })
        continue

    try:
        batch_rows.append(
            collect_ticker_dataset(ticker, risk_free_rate, risk_free_source)
        )
    except Exception as error:
        print(f"  Failed: {type(error).__name__}: {error}")
        batch_rows.append({
            "ticker": ticker,
            "status": "failed",
            "spot_price": None,
            "raw_contracts": None,
            "usable_contracts": None,
            "filter_mode": None,
            "technical_signal": None,
            "output_directory": str(export_paths["success_marker"].parent),
            "error": f"{type(error).__name__}: {error}",
        })

    if REQUEST_PAUSE_SECONDS > 0 and position < len(tickers):
        time.sleep(REQUEST_PAUSE_SECONDS)

batch_report = pd.DataFrame(batch_rows)
batch_report_path = OUTPUT_ROOT / "batch_report.csv"
batch_report.to_csv(batch_report_path, index=False)

display(batch_report)
print("\nStatus summary")
display(batch_report["status"].value_counts().rename_axis("status").to_frame("tickers"))
print(f"Batch report: {batch_report_path}")


## 6. Optional visual check

Visualisation is limited to one completed ticker so an S&P 500 batch does not create hundreds of charts.


In [ ]:
completed_tickers = batch_report.loc[
    batch_report["status"].eq("completed"), "ticker"
].tolist()

if DISPLAY_SAMPLE and completed_tickers:
    sample_ticker = completed_tickers[0]
    sample_paths = expected_export_paths(sample_ticker)
    sample_history = pd.read_csv(
        sample_paths["history_features"], parse_dates=["Date"], index_col="Date"
    )
    sample_options = pd.read_csv(sample_paths["option_universe"])

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    sample_history[["Close", "sma_20", "sma_50"]].plot(
        ax=axes[0], title=f"{sample_ticker} price and trend features"
    )
    axes[0].set_ylabel("Price (USD)")
    axes[0].grid(alpha=0.25)

    for option_type, group in sample_options.groupby("option_type"):
        axes[1].scatter(
            group["strike"],
            group["impliedVolatility"],
            label=option_type.title(),
            alpha=0.65,
            s=24,
        )
    spot = float(sample_options["spot_price"].iloc[0])
    axes[1].axvline(
        spot, color="black", linestyle="--", linewidth=1, label="Spot"
    )
    axes[1].set_title(f"{sample_ticker} current implied-volatility smile")
    axes[1].set_xlabel("Strike")
    axes[1].set_ylabel("Implied volatility")
    axes[1].legend()
    axes[1].grid(alpha=0.25)

    plt.tight_layout()
    plt.show()
else:
    print("Visual sample skipped or no ticker completed in this run.")


## 7. Output structure and hand-off

```text
outputs/notebook_01/
├── sp500_constituents.csv       # S&P 500 mode only
├── batch_report.csv
├── CRM/
│   ├── history_features.csv
│   ├── option_chain_raw.csv
│   ├── option_universe.csv
│   ├── market_snapshot.json
│   ├── quality_report.csv
│   └── _SUCCESS.json
└── AAPL/
    └── ...
```

Notebook 2 should read ticker data from `outputs/notebook_01/<TICKER>/`. Notebook 3 should introduce `OUTLOOK = "bullish"` or `"bearish"` and write outlook-specific results under `outputs/<TICKER>/<OUTLOOK>/`.


In [ ]:
data_dictionary = pd.DataFrame([
    ("contractSymbol", "Unique OCC option contract identifier"),
    ("option_type", "call or put; both are retained for every ticker"),
    ("expiry / dte", "Contract expiry and calendar days to expiry"),
    ("mid_price", "Bid-ask midpoint, falling back to last price for an invalid quote"),
    ("bid_ask_spread_pct", "Relative transaction-cost/liquidity proxy"),
    ("strike_to_spot", "Strike divided by current stock price"),
    ("moneyness_pct", "Signed strike distance from spot"),
    ("impliedVolatility", "Annualised implied volatility from the option chain"),
    ("delta / gamma / theta_day / vega_1pct", "Black-Scholes sensitivity estimates"),
    ("liquidity_pass", "Whether open-interest and spread rules are satisfied"),
    ("market_snapshot.json", "Observed facts and assumptions; no directional user outlook"),
    ("batch_report.csv", "Completed, skipped and failed tickers with diagnostic messages"),
], columns=["field", "meaning"])
display(data_dictionary)


## Notebook 1 completion criteria

Notebook 1 is complete when:

- the S&P 500 constituent snapshot is saved in S&P 500 mode;
- each successful ticker has the five validated data files and a `_SUCCESS.json` marker;
- `batch_report.csv` records every completed, skipped or failed ticker; and
- both calls and puts are available for downstream bullish and bearish reasoning.
